# LLM-assisted petroleum review

## text-to-SQL

This notebook demonstrates a controlled text-to-SQL workflow for petroleum review data. It uses a public demo context extracted from FieldViewer Y1 materials and runs fully offline with deterministic SQL generation.

## 1. Load the demo module and build the database

The demo context is converted into an in-memory SQLite database with safe AI-facing views.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from llm_petroleum_review.database import build_demo_database, build_schema_context
from llm_petroleum_review.text_to_sql import answer_question, generate_sql, execute_validated_sql
from llm_petroleum_review.sql_guard import validate_sql

connection = build_demo_database(PROJECT_ROOT / 'data' / 'y1_ai_context')
connection.execute('SELECT COUNT(*) AS well_count FROM wells').fetchone()['well_count']

## 2. Inspect the AI-visible schema context

The LLM should receive compact schema metadata, not a direct database handle.

In [ ]:
schema_context = build_schema_context(connection)
[(table['table_name'], table['grain']) for table in schema_context['tables']]

## 3. Ask a petroleum review question

In production, an LLM can produce the candidate SQL from the schema context. This public demo uses deterministic patterns so it can be run without API keys.

In [ ]:
question = 'List all wells with coordinates'
candidate = generate_sql(question, schema_context)
candidate

## 4. Validate before execution

Only `SELECT` and `WITH` statements are allowed. Missing limits are added automatically.

In [ ]:
validation = validate_sql(candidate['sql'], connection=connection)
validation

## 5. Execute the validated SQL

The execution layer uses the validator output, not the raw model text.

In [ ]:
result = execute_validated_sql(connection, candidate['sql'])
result['row_count'], result['rows'][:3]

## 6. Run the complete answer workflow

The final answer is summarized only from returned SQL rows.

In [ ]:
response = answer_question(connection, 'How many wells are there by reservoir?')
print(response['generated_sql'])
print()
print(response['answer'])

## 7. Show unsafe SQL rejection

Dangerous SQL is rejected and never executed.

In [ ]:
unsafe_examples = [
    'DROP TABLE wells',
    'SELECT * FROM wells; DELETE FROM wells',
    'SELECT * FROM audit_log',
    'SELECT * FROM wells -- DELETE FROM wells',
]

for sql in unsafe_examples:
    print(sql, '->', validate_sql(sql, connection=connection)['message'])

## 8. Try additional questions

These examples map to the same safe workflow.

In [ ]:
questions = [
    'What layers are available?',
    'Which wells have production context?',
    'Summarize wells by status',
]

for q in questions:
    response = answer_question(connection, q)
    print('\nQUESTION:', q)
    print('SQL:', response['generated_sql'])
    print(response['answer'].split('\n')[0])